# ACS ECG Detector — обучение на Google Colab (GPU T4)

Выполняет Этапы 4-5 (CNN + Multimodal) на бесплатном GPU T4.

## Перед запуском
1. Убедиться, что `processed.zip` загружен на Yandex.Disk (ссылка уже в коде)
2. Runtime -> Change runtime type -> T4 GPU

In [ ]:
# Ячейка 1: Установка зависимостей (если не установлены)
import sys, subprocess, os

# Клонировать репозиторий, если не склонирован
REPO_URL = "https://github.com/mlmamalyga83/acs-risk-ai.git"
if not os.path.exists('acs-risk-ai'):
    !git clone {REPO_URL}

%cd acs-risk-ai
!pip install -r requirements.txt -q

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Ячейка 2: Скачать processed.zip с Yandex.Disk
import urllib.request, urllib.parse, json

PUBLIC_KEY = "https://disk.yandex.ru/d/03GsU4NnX2RpEQ"
api_url = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={urllib.parse.quote(PUBLIC_KEY)}"

print("Getting download link from Yandex.Disk...")
with urllib.request.urlopen(api_url) as resp:
    download_url = json.loads(resp.read())["href"]

!wget -O processed.zip "{download_url}" --show-progress
!unzip -o processed.zip -d /content/
print('OK processed.zip extracted to /content/')

In [ ]:
# Ячейка 3: Скопировать данные и конфиги в репозиторий
# processed.zip распакован в /content/data/processed/ и /content/config/
!cp -r /content/data/ ./ 2>/dev/null || echo "data copy done"
!cp -r /content/config/ ./ 2>/dev/null || echo "config copy done"

print('OK data and config copied')

In [ ]:
# Ячейка 4: Проверить GPU и данные
import torch
assert torch.cuda.is_available(), "GPU не обнаружен! Runtime -> Change runtime type -> T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

import numpy as np
proc = 'data/processed'
print(f"Train files: {open(f'{proc}/X_train_manifest.txt').read().count(chr(10))} batches")
print(f"Val:   {np.load(f'{proc}/X_val.npy', mmap_mode='r').shape}")
print(f"Test:  {np.load(f'{proc}/X_test.npy', mmap_mode='r').shape}")
print(f"Train: {np.load(f'{proc}/y_train.npy').shape}, ACS={np.load(f'{proc}/y_train.npy').mean()*100:.1f}%")
print('OK all checks passed')

In [ ]:
# Ячейка 5: CNN обучение с Optuna (~6 часов на T4)
!python run.py --stage cnn --tune

In [ ]:
# Ячейка 6: Multimodal обучение + Ablation (~4 часа на T4)
!python run.py --stage multimodal --tune

In [ ]:
# Ячейка 7: Сохранить результаты — упаковать и скачать
import os, tarfile

models = [f for f in os.listdir('models') if f.endswith('.pt')]
print(f'Models trained: {len(models)}')
for m in models:
    sz = os.path.getsize(f'models/{m}') / 1e6
    print(f'  {m}: {sz:.1f} MB')

# Сохранить в корень Colab для скачивания
with tarfile.open('/content/models.tar.gz', 'w:gz') as tar:
    for m in models:
        tar.add(f'models/{m}')
print('OK /content/models.tar.gz created')
print('\nНажмите на папку /content/ в боковой панели Colab,')
print('найдите models.tar.gz, кликните ПКМ -> Download')